## Step 1: Load each CSV

In [4]:
import pandas as pd

categories = pd.read_csv('categories.csv')
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
order_details = pd.read_csv('order_details.csv')
products = pd.read_csv('products.csv')

## Step 2: Look at each one individually

In [12]:
for name, df in [('categories', categories), ('customers', customers),
                ('orders', orders), ('order_details', order_details),
                ('products', products)]:
    print(f"--- {name} ---")
    print(df.shape)
    print(df.columns.tolist())
    print(df.head(3))
    print()

--- categories ---
(10, 2)
['CategoryID', 'CategoryName']
  CategoryID    CategoryName
0  CTGRY0001  Meat & Poultry
1  CTGRY0002    Dairy & Eggs
2  CTGRY0003         Produce

--- customers ---
(1000, 7)
['CustomerID', 'Gender', 'Age', 'City', 'Region', 'CustomerSegment', 'SignUpDate']
  CustomerID  Gender  Age      City   Region CustomerSegment  SignUpDate
0  CSTMR0001    Male   29  Istanbul  Marmara        Standard  2020-04-13
1  CSTMR0002   Other   40  Istanbul  Marmara        Standard  2020-04-08
2  CSTMR0003  Female   36     Bursa  Marmara         Premium  2022-08-14

--- orders ---
(10000, 4)
['OrderID', 'CustomerID', 'OrderDate', 'OrderTime']
    OrderID CustomerID   OrderDate OrderTime
0  ORD00001  CSTMR0983  2021-04-10  12:53:24
1  ORD00002  CSTMR0703  2024-04-21  22:08:16
2  ORD00003  CSTMR0920  2022-10-27  10:21:26

--- order_details ---
(30271, 10)
['OrderID', 'ProductID', 'Quantity', 'UnitCost', 'UnitPrice', 'DiscountRate', 'IsReturned', 'ReturnDate', 'ReturnTime', 'ReturnR

## Step 3: Also open the .db file -- this uses your SQL knowledge directly

In [16]:
import sqlite3

# This opens a connection to the .df file(similar to connecting to MySQL)
conn = sqlite3.connect('retail_store.db')

# See what tables exist inside it
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)
print(tables)

            name
0     categories
1       products
2      customers
3         orders
4  order_details


### Step explained

- **sqlite3.connect('retail_store.db')** -- this opens a connection to the .db file, similar to how you'd connect to MySQL with a username/password, except SQLite databases are just a single file on disk -no server, no login needed.conn is now your "pipe" to that database.
- **sqlite_master** -- this is a special built-in table that every SQLite database automatically has. It stores metadata about listing of every table, index, and view inside it.
- **pd.read_sql(...)** -- instead of manually running the query and looping through results, this pandas functiion runs the SQL query and immediately hands you back the result as a dataframe -- combining SQL and pandas in one step

## Step 4: Peek at each table's structure using SQL

In [14]:
query = """
SELECT * FROM orders LIMIT 5;
"""

pd.read_sql(query, conn)

,OrderID,CustomerID,OrderDate,OrderTime
0,ORD00001,CSTMR0983,2021-04-10,12:53:24
1,ORD00002,CSTMR0703,2024-04-21,22:08:16
2,ORD00003,CSTMR0920,2022-10-27,10:21:26
3,ORD00004,CSTMR0158,2025-11-07,22:24:27
4,ORD00005,CSTMR0834,2021-09-05,13:23:16


### Step explained  

- **SELECT * FROM orders** -- standard SQL: grab every column from the orders table.
- **LIMIT 5** -- only return 5 rows, just to peek at the structure without pulling the entire table.
- **pd.read_sql(query, conn)** -- runs the query against the same connection from Step 3, and returns it as a pandas dataframe you can view/manipulate normally.

The purpose: once we know a table's name (from Step 3), we look inside it to see its actual columns and sample data -- this tells us how it might connect to the CSV files you already loaded (e.g,**if orders has a customer_id** column, that's the link to your **customers.csv**)